# NABBP DATA - EDA (WIP by Cyril)

In [ ]:
import altair as alt

from birds.source_data.nabbp import LookupTables, DataTables
from birds.settings import load_settings

In [ ]:
alt.data_transformers.enable("vegafusion")

In [ ]:
settings = load_settings()

## Species Index

In [ ]:
data = DataTables()
lookup = LookupTables()

species_df = data.species_index

In [ ]:
print('Unique species count in the dataset: ', species_df['species_id'].nunique())

In [ ]:
def plot_top_species_bar_chart(df, top_n=20):
    top_species = df[['SPECIES_NAME', 'count']].sort_values(by='count', ascending=False).head(top_n)    
    chart = (
        alt.Chart(top_species)
            .mark_bar()
            .encode(
                x=alt.X('count:Q', title='Count'),
                y=alt.Y('SPECIES_NAME:N', sort='-x', title='Species Name'),
                tooltip='count:N'
            )
            .properties(
                title=f'Top {top_n} Species by Count in NABBP Data',
            )
    )
    return chart


plot_top_species_bar_chart(species_df, top_n=20)

## Selected Species Stats

## 

In [ ]:
lazy_frame = data.load_data_by_species_names(
    names=[
        'Mallard',
        #'Song Sparrow',
    ]
)

In [ ]:
unique_bands_count = lazy_frame.select('band').unique().count().collect().item()
print("Unique bands: ", unique_bands_count)

In [ ]:

def plot_num_captures_distribution(frame):
    encounter_counts = (
        frame
            .group_by('band')
            .len(name='num_captures')
            .collect()
            .to_pandas()
    )

    chart = (
        alt.Chart(encounter_counts)
            .mark_bar()
            .encode(
                x=alt.X('num_captures:Q', bin=alt.Bin(maxbins=30, step=1), title='Number of Captures'),
                y=alt.Y('count():Q', title='Number of Bands'),
                tooltip=['count():Q']
            )
            .properties(
                title='Distribution of Number of Captures per Band',
            )
    )
    return chart

plot_num_captures_distribution(lazy_frame)